In [1]:
using CSV, DataFrames, Statistics, Printf, StatsBase, Plots

include("functions.jl")

data_dir  = "output"
burnin    = 500_000
thin      = 10
chain_ids = 1:3
outfile   = joinpath(data_dir, "rhat_summary.csv")

filename_for_chain(c) = joinpath(data_dir, "samples_chain_$(c).csv")

function load_chain_df(path::String; burnin::Int, thin::Int)
    df = CSV.read(path, DataFrame)
    idx = (burnin + 1):thin:nrow(df)
    return df[idx, :]
end

dfs = DataFrame[]
for c in chain_ids
    fpath = filename_for_chain(c)
    push!(dfs, load_chain_df(fpath; burnin=burnin, thin=thin))
end

n_keep = minimum(nrow.(dfs))
dfs = [df[1:n_keep, :] for df in dfs]

params = String.(names(dfs[1]))
params = [p for p in params if eltype(dfs[1][!, p]) <: Real]

rhat_vals = Float64[]
rhat_names = String[]

for p in params
    mat = Array{Float64}(undef, length(chain_ids), n_keep)
    for (i, df) in enumerate(dfs)
        mat[i, :] = Float64.(df[!, p])
    end
    push!(rhat_names, p)
    push!(rhat_vals, rhat_gelman_rubin(mat))
end

outdf = DataFrame(param = rhat_names, Rhat = rhat_vals)
CSV.write(outfile, outdf)

println("Wrote Rhat summary to: $outfile")


Wrote Rhat summary to: output/rhat_summary.csv


In [ ]:
using DelimitedFiles
using Statistics
using StatsBase
using Random
using Printf


const OUTPUT_DIR        = "output"
const N_CHAINS          = 3
const BURN_IN_SAMPLES   = 500_000    
const THIN_SAMPLES      = 10    

const BURN_IN_LOGLIK    = 0
const THIN_LOGLIK       = 1

param_header = ["beta","alpha","gamma"]

const N_DRAWS = 1000

function read_csv_matrix(path::String)
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    M = Matrix{Float64}(data)
    header = hdr === nothing ? nothing : vec(collect(hdr))
    return M, header
end

function load_samples_for_chain(c::Int, outdir::String)
    path = joinpath(outdir, "samples_chain_$(c).csv")
    if !isfile(path)
        @warn "Missing samples file: $path"; return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if size(M,1) <= BURN_IN_SAMPLES
        @warn "File $path has only $(size(M,1)) rows (<= burn-in)."
        return Array{Float64}(undef, 0, 0)
    end
    idxs = collect(BURN_IN_SAMPLES+1:THIN_SAMPLES:size(M,1))
    return M[idxs, :]
end

function load_loglik_for_chain(c::Int, outdir::String)
    path = joinpath(outdir, "loglik_chain_$(c).csv")
    if !isfile(path)
        @warn "Missing loglik file: $path"; return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if BURN_IN_LOGLIK > 0 || THIN_LOGLIK > 1
        idxs = collect(BURN_IN_LOGLIK+1:THIN_LOGLIK:size(M,1))
        M = M[idxs, :]
    end
    return M
end

function logmeanexp_columnwise(L::AbstractMatrix{<:Real})
    S, T = size(L)
    out = Vector{Float64}(undef, T)
    for j in 1:T
        col = L[:, j]
        m = maximum(col)
        out[j] = m + log(sum(exp.(col .- m)) / S)
    end
    return out
end

function compute_waic_from_list(all_loglik_aug_vecs::Vector{Matrix{Float64}})
    combined_logliks = vcat(all_loglik_aug_vecs...)
    if any(isinf, combined_logliks)
        @warn "Encountered infinite log-likelihoods. WAIC will be -Inf."
        return -Inf
    end
    lppd = sum(logmeanexp_columnwise(combined_logliks))
    pwaic = sum(var(combined_logliks, dims=1))
    return -2 * lppd + 2 * pwaic
end

function summarize_posterior(samples::AbstractMatrix{<:Real}, header_cont::Vector{String})
    summary_dict = Dict{String, Any}()
    for (i, p_name) in enumerate(header_cont)
        param_samples = samples[:, i]
        if p_name == "k_max"
            k = mode(round.(Int, param_samples))
            freq = count(==(k), round.(Int, param_samples)) / length(param_samples)
            summary_dict[p_name] = Dict("mode" => k, "frequency" => freq)
        else
            med = median(param_samples)
            q = quantile(param_samples, [0.025, 0.975])
            summary_dict[p_name] = Dict("median" => med, "95%CI_low" => q[1], "95%CI_high" => q[2])
        end
    end
    return summary_dict
end

function write_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        DelimitedFiles.writedlm(io, M, ',')
    end
end

function write_summary_csv(path::String, summary_dict::Dict{String,Any})
    open(path, "w") do io
        println(io, "Parameter,Median_or_Mode,CI_Low_or_Frequency,CI_High")
        for (param, vals) in summary_dict
            if haskey(vals, "mode")
                println(io, "$param,$(vals["mode"]),$(vals["frequency"]),")
            else
                println(io, "$param,$(vals["median"]),$(vals["95%CI_low"]),$(vals["95%CI_high"])")
            end
        end
    end
end

Random.seed!(2025)

samples_all = Matrix{Float64}[]
for c in 1:N_CHAINS
    push!(samples_all, load_samples_for_chain(c, OUTPUT_DIR))
end
samples_bt = vcat(samples_all...)
if size(samples_bt,1) == 0
    error("No samples found")
end

posterior_summary = summarize_posterior(samples_bt, param_header)

ll_list = Matrix{Float64}[]
for c in 1:N_CHAINS
    M = load_loglik_for_chain(c, OUTPUT_DIR)
    if size(M,1) > 0
        push!(ll_list, M)
    end
end
waic_val = isempty(ll_list) ? NaN : compute_waic_from_list(ll_list)

n_rows = size(samples_bt, 1)
replace_flag = n_rows < N_DRAWS
draw_indices = sample(1:n_rows, N_DRAWS; replace=replace_flag)
draw_params  = samples_bt[draw_indices, :]

draws_outfile = joinpath(OUTPUT_DIR, @sprintf("posterior_draws.csv"))
hdr = vcat(["row_index"], param_header)
Mout = hcat(Float64.(draw_indices), draw_params)
write_csv(draws_outfile, hdr, Mout)

write_summary_csv(joinpath(OUTPUT_DIR, "posterior_summary.csv"), posterior_summary)

open(joinpath(OUTPUT_DIR, "waic.csv"), "w") do io
    println(io, "WAIC")
    println(io, waic_val)
end

@info "Done. Outputs written in $(OUTPUT_DIR)"
#WAIC: 1043.54

[ Info: Done. Outputs written in output
